# Stable Diffusion 1.5 — примеры использования

Три сценария: быстрый запуск в 2 строчки, полная сборка графа, добавление отдельных компонентов.

## 1. Пользовательское использование в 2 строчки кода

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_text2img", device="cuda")
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars", 
    num_inference_steps=30, 
    guidance_scale=7.5, 
    seed=42, 
    width=512, 
    height=512
)

img = output.images[0] if output.images else None
img

## 2. Полная сборка графа модели

Собираем граф вручную из компонентов SD1.5 с автосвязыванием. Служебные ноды (latent_init и т.д.) подставляются автоматически.

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph(name="MySD15Graph")
pretrained = "runwayml/stable-diffusion-v1-5"

graph.add_node("Backbone", type="sd15.backbone", pretrained=pretrained)
graph.add_node("Tokenizer", type="sd15.tokenizer", pretrained=pretrained)
graph.add_node("TextEncoder", type="sd15.text_encoder", pretrained=pretrained)
graph.add_node("Scheduler", type="sd15.scheduler", pretrained=pretrained)
graph.add_node("Autoencoder", type="sd15.autoencoder", pretrained=pretrained)

graph.infer_exposed_ports()
graph.to("cuda")

In [ ]:
# Запуск собранного графа
output = graph.run(prompt="a cat in a spacesuit", negative_prompt="blurry", num_inference_steps=25, guidance_scale=7.5, seed=123, width=512, height=512)
output.images[0] if output.images else None

## 3. Добавление отдельных компонентов

Берём готовый граф из шаблона и добавляем адаптеры (ControlNet, LoRA) или заменяем компоненты по имени.

### 3.1 Замена компонентов в готовом графе

In [ ]:
graph = Hypergraph.from_template("sd15_text2img", device="cuda")

# Меняем веса на другую модель (например, fine-tune)
new_repo = "runwayml/stable-diffusion-v1-5"
graph.replace_node("prompt_enc", pretrained=new_repo)
graph.replace_node("unet", pretrained=new_repo)
graph.replace_node("vae_decode", pretrained=new_repo)

### 3.2 Добавление ControlNet к готовому графу

In [ ]:
graph = Hypergraph.from_template("sd15_text2img", device="cuda")

# ControlNet подключается к backbone автоматически
graph.add_node("CannyControl", type="adapter.controlnet", pretrained="lllyasviel/sd-controlnet-canny")

### 3.3 Сборка графа по частям: только нужные компоненты

Можно добавлять компоненты по одному (tokenizer, prompt_encoder, backbone, scheduler, vae). Порядок добавления задаёт автосвязывание по именам портов.

In [ ]:
from yggdrasill import Hypergraph

g = Hypergraph(name="SD15FromScratch")
repo = "runwayml/stable-diffusion-v1-5"

# Отдельные компоненты — каждый add_node подтягивает связи к уже существующим узлам
g.add_node("tok", type="sd15.tokenizer", pretrained=repo)
g.add_node("enc", type="sd15.prompt_encoder", pretrained=repo)
g.add_node("unet", type="sd15.backbone", pretrained=repo)
g.add_node("sched", type="sd15.scheduler", pretrained=repo)
g.add_node("vae", type="sd15.autoencoder", pretrained=repo)

g.infer_exposed_ports()
g.to("cuda")

In [ ]:
out = g.run(prompt="red apple on white background", num_inference_steps=20, guidance_scale=7.5, seed=1, width=512, height=512)
out.images[0] if out.images else None